## Load Developed Countries Across The World

In [1]:
import plotly.express as px
import plotly.io as pio

from developed_country import get_dev_df
dev_df = get_dev_df()

fig = px.choropleth(
    dev_df,
    locations="iso3",
    color="group",
    projection="natural earth",
    title="Developed Countries Map",
    width=1000,
    height=1000
)

fig.show()

## Load English Speaking Countries

In [2]:
english_iso3 = [
    # Native English
    "USA","CAN","GBR","AUS","NZL","IRL",

    # Strong English (Europe)
    "MLT","NLD","SWE","NOR","DNK","FIN",

    # English official/common
    "SGP"
]

df_english = dev_df[dev_df["iso3"].isin(english_iso3)].copy()

print("Total English-Speaking Developed Countries:", len(df_english))
df_english

fig = px.choropleth(
    df_english,
    locations="iso3",
    color="group",
    projection="natural earth",
    title="English-Speaking Countries Map",
    width=1000,
    height=1000
)

fig.show()

Total English-Speaking Developed Countries: 12


## Download URL

In [3]:
from download_pipeline import download_all
from master_builder import build_master

# 1) download everything
download_all()

# 2) get english ISO3 list in dataframe format
selected_iso3 = df_english.rename(columns={"ISO":"iso3"})[["iso3"]].drop_duplicates()

# 3) define metrics list (same as you already have)
metrics = [
    ("econ_gdp_percap.csv", None, "gdp_percap"),
    ("econ_inflation_cpi.csv", None, "inflation_cpi"),
    ("econ_price_level_ratio_vs_us.csv", None, "price_level_ratio"),
    ("social_homicide_rate.csv", None, "homicide_rate"),
    ("social_political_stability.csv", None, "political_stability"),
    ("social_freedom_expression.csv", None, "freedom_expression"),
    ("health_life_expectancy.csv", None, "life_expectancy"),
    ("health_physicians_per_1000.csv", None, "physicians_per_1000"),
    ("health_spend_percap.csv", None, "health_spend_percap"),
    ("edu_tertiary_enroll.csv", None, "tertiary_enrollment"),
    ("edu_rd_percent_gdp.csv", None, "rd_percent_gdp"),
    ("edu_patents_resident.csv", None, "patents_resident"),
    ("edu_population.csv", None, "population"),
    ("env_pm25.csv", None, "pm25"),
    ("env_renewable_share.csv", None, "renewable_share"),
    ("env_ndgain_gain.csv", None, "ndgain_gain"),
    ("env_ndgain_readiness.csv", None, "ndgain_readiness"),
    ("env_ndgain_vulnerability.csv", None, "ndgain_vulnerability"),
    ("infra_internet_users.csv", None, "internet_users_pct"),
    ("infra_rail_km.csv", None, "rail_km"),
    ("infra_road_deaths_per_100k.csv", None, "road_deaths_per_100k"),
    ("legal_rule_of_law.csv", None, "rule_of_law"),
    ("legal_gov_effectiveness.csv", None, "gov_effectiveness"),
    ("legal_political_stability.csv", None, "legal_political_stability"),
    ("culture_migrant_share.csv", None, "migrant_share"),
    ("culture_migrant_total.csv", None, "migrant_total"),
]

# 4) build master
master = build_master(selected_iso3, metrics)
master.head()

Extracted resources/gain/gain.csv -> data_raw/env_ndgain_gain.csv (rows=187)
Extracted resources/readiness/readiness.csv -> data_raw/env_ndgain_readiness.csv (rows=192)
Extracted resources/vulnerability/vulnerability.csv -> data_raw/env_ndgain_vulnerability.csv (rows=187)
Master saved: data_processed/master.csv | rows: 12 | cols: 28


,iso3,gdp_percap,inflation_cpi,price_level_ratio,homicide_rate,political_stability,freedom_expression,life_expectancy,physicians_per_1000,health_spend_percap,...,ndgain_vulnerability,internet_users_pct,rail_km,road_deaths_per_100k,rule_of_law,gov_effectiveness,legal_political_stability,migrant_share,migrant_total,patents_per_million
0,USA,84534.040784,2.949525,1.000000,5.763408,0.029425,0.889,78.385366,3.681,13473.193359,...,0.316926,93.1444,148553.34734,12.7,1.327678,1.217201,0.029425,15.162426,52375047.0,771.054183
1,CAN,54340.347722,2.381584,0.840153,1.979689,0.822421,0.943,81.646585,2.819,6377.968750,...,0.287536,93.9564,48149.90622,5.3,1.473006,1.516803,0.822421,22.157274,8805839.0,114.075074
2,DNK,71026.483227,1.372200,0.877544,0.840599,0.850848,0.985,81.853659,4.498,6745.061523,...,0.344710,99.7692,2131.00000,3.7,1.908936,2.015649,0.850848,14.177959,847475.0,182.365979
3,FIN,53149.767193,1.565689,0.816055,0.981935,0.713987,0.952,81.685366,3.609,5515.404785,...,0.291436,93.5139,5918.00000,3.9,1.967046,1.738607,0.713987,9.157978,514432.0,277.050651
4,MLT,43898.578181,1.650795,0.628346,0.562898,0.858122,0.838,83.507317,4.513,3623.460938,...,0.326815,92.0727,NaN,4.1,0.702328,0.397335,0.858122,36.965050,199466.0,8.789710


In [9]:
import importlib
import scoring_and_mapping as sm

# Reload to reflect any edits you made in scoring_and_mapping.py
importlib.reload(sm)

# Compute scores
factor_scores = sm.score_countries(master)

# Export ranking
factor_scores.to_csv("data_processed/country_scores.csv", index=False)

# Show leaderboard
display(factor_scores[["rank", "iso3", "final_score"]])

# Plot map
fig = sm.make_discrete_map(factor_scores)
fig.show()

,rank,iso3,final_score
0,1,SGP,1.000000
1,2,DNK,0.881438
2,3,NOR,0.874449
3,4,AUS,0.865069
4,5,FIN,0.855992
5,6,USA,0.850996
6,7,SWE,0.823240
7,8,CAN,0.788751
8,9,NZL,0.778435
9,10,MLT,0.777178
